# Backtest UAS Algoritma Trading (Colab)
Jalankan sel dari atas ke bawah. Sel 1 membuat `_core.py` (salinan `api/_core.py` di repo, parameternya harus sama). Sel 3 menarik data Indodax (tidak paralel), sel 4 mengukur spread dan slippage dari log bot, sel 5 menjalankan backtest, sel 6 mengemas hasil.

In [ ]:
%%writefile _core.py
"""Logika inti BERSAMA: dipakai bot (api/tick.py) DAN backtest (Colab).
Hanya Python standar (tanpa pandas) supaya ringan di Vercel.
Parameter di bagian atas = DIBEKUKAN saat freeze date."""
import math
from datetime import datetime, timezone, timedelta

# ---------- Parameter strategi (jangan diubah setelah freeze) ----------
TF_SECONDS = 4 * 3600          # timeframe 4 jam
TF_API = "240"                 # nilai tf untuk history_v2
ENTRY_N = 20                   # masuk: close > high tertinggi 20 bar sebelumnya
EXIT_K = 10                    # keluar: close < low terendah 10 bar sebelumnya
SMA_M = 100                    # filter: close > SMA 100 (keluar juga bila di bawah SMA)
BARS_NEEDED = max(SMA_M, ENTRY_N + 1, EXIT_K + 1)
UNIVERSE = ["btcidr", "ethidr", "adaidr"]

INITIAL_CASH = 100_000_000
MAX_POSITIONS = 4
MAX_POS_FRAC = 0.20            # maks 20% equity per posisi
DAILY_LOSS_LIMIT = 0.03        # 3% equity, reset 00.00 WIB
MIN_ORDER_IDR = 25_000         # pakai yang lebih ketat (Pro); cek halaman resmi
MAX_SPREAD = 0.02              # spread bid-ask > 2% = data tidak valid
QTY_DECIMALS = 8               # TODO: samakan dengan presisi pair di /api/pairs


# ---------- Waktu ----------
def to_seconds(ts):
    ts = float(ts)
    return int(ts / 1000) if ts > 1e12 else int(ts)


def wib_date(ts_seconds):
    d = datetime.fromtimestamp(ts_seconds, tz=timezone.utc) + timedelta(hours=7)
    return d.strftime("%Y-%m-%d")


# ---------- Data candle ----------
def normalize_bars(raw):
    """Ubah respons history_v2 jadi list dict {t,o,h,l,c,v}, urut naik.
    CEK bentuk respons asli Indodax; fungsi ini menerima dict ber-key
    Time/Open/High/Low/Close/Volume atau list [t,o,h,l,c,v]."""
    if isinstance(raw, dict):
        raw = raw.get("data") or raw.get("candles") or []
    out = []
    for r in raw:
        if isinstance(r, dict):
            def g(*ks):
                for k in ks:
                    if k in r and r[k] is not None:
                        return r[k]
                raise ValueError("kolom candle tidak ditemukan: %s" % (ks,))
            t, o, h, l, c, v = (g("Time", "time", "t"), g("Open", "open", "o"),
                                g("High", "high", "h"), g("Low", "low", "l"),
                                g("Close", "close", "c"), g("Volume", "volume", "v"))
        else:
            t, o, h, l, c, v = r[:6]
        out.append({"t": to_seconds(t), "o": float(o), "h": float(h),
                    "l": float(l), "c": float(c), "v": float(v)})
    out.sort(key=lambda b: b["t"])
    return out


def drop_open_bars(bars, server_time_s, tf=TF_SECONDS):
    """Buang bar yang masih berjalan: Time + tf > server_time."""
    return [b for b in bars if b["t"] + tf <= server_time_s]


def parse_levels(raw, desc):
    out = []
    for lvl in raw or []:
        p, a = float(lvl[0]), float(lvl[1])
        if p > 0 and a > 0:
            out.append((p, a))
    out.sort(key=lambda x: x[0], reverse=desc)
    return out


def validate(bars, bids, asks, server_time_s, tf=TF_SECONDS):
    """Return (ok, [alasan]). Kalau tidak ok, bot tidak boleh membuka posisi."""
    reasons = []
    if len(bars) < BARS_NEEDED:
        reasons.append("bar_kurang(%d<%d)" % (len(bars), BARS_NEEDED))
    else:
        win = bars[-BARS_NEEDED:]
        # umur dihitung dari waktu TUTUP candle terakhir
        if server_time_s - (win[-1]["t"] + tf) > 2 * tf:
            reasons.append("candle_terakhir_basi")
        if any(win[i + 1]["t"] - win[i]["t"] != tf for i in range(len(win) - 1)):
            reasons.append("candle_hilang")
        if any(b["h"] <= 0 or b["l"] <= 0 or b["c"] <= 0 or b["v"] <= 0 for b in win):
            reasons.append("harga_atau_volume_nol")
    if not bids or not asks:
        reasons.append("order_book_kosong")
    else:
        spread = (asks[0][0] - bids[0][0]) / asks[0][0]
        if spread > MAX_SPREAD:
            reasons.append("spread_%.2f%%" % (spread * 100))
    return (len(reasons) == 0), reasons


# ---------- Sinyal (dipakai backtest & bot, sama persis) ----------
def compute_signal(bars, in_position):
    """bars = HANYA candle tertutup, urut naik. Return dict berisi 'signal'."""
    if len(bars) < BARS_NEEDED:
        return {"signal": "NONE", "why": "bar_kurang"}
    last = bars[-1]
    sma = sum(b["c"] for b in bars[-SMA_M:]) / SMA_M
    hh = max(b["h"] for b in bars[-1 - ENTRY_N:-1])
    ll = min(b["l"] for b in bars[-1 - EXIT_K:-1])
    sig = "HOLD"
    if not in_position:
        if last["c"] > hh and last["c"] > sma:
            sig = "BUY"
    else:
        if last["c"] < ll or last["c"] < sma:
            sig = "SELL"
    return {"signal": sig, "close": last["c"], "hh": hh, "ll": ll, "sma": sma,
            "candle_t": last["t"]}


# ---------- Eksekusi di order book (bukan mid/close) ----------
def floor_qty(q):
    f = 10 ** QTY_DECIMALS
    return math.floor(q * f + 1e-9) / f


def vwap_for_qty(levels, qty):
    need, cost = qty, 0.0
    for p, a in levels:
        take = min(a, need)
        cost += take * p
        need -= take
        if need <= 1e-12:
            break
    return cost, qty - max(need, 0.0)


def simulate_buy(asks, budget_idr, fee):
    """Beli di ask, habiskan level berikutnya bila perlu. Fee = tarif taker all-in."""
    remaining, qty = budget_idr / (1 + fee), 0.0
    for p, a in asks:
        spend = min(a * p, remaining)
        qty += spend / p
        remaining -= spend
        if remaining <= 1e-9:
            break
    qty = floor_qty(qty)
    if qty <= 0:
        return None
    cost, filled = vwap_for_qty(asks, qty)
    if filled < qty - 1e-12:
        return None
    fee_idr = cost * fee
    return {"qty": qty, "avg": cost / qty, "cost": cost,
            "fee": fee_idr, "total": cost + fee_idr}


def simulate_sell(bids, qty, fee):
    """Jual di bid. None bila kedalaman order book tidak cukup."""
    cost, filled = vwap_for_qty(bids, qty)
    if filled < qty - 1e-12:
        return None
    fee_idr = cost * fee
    return {"qty": qty, "avg": cost / qty, "gross": cost,
            "fee": fee_idr, "net": cost - fee_idr}


In [ ]:
# SEL 1b: cek parameter (harus sama dengan README / repo)
import sys; sys.path.insert(0, ".")
import importlib, _core as core; importlib.reload(core)
print("UNIVERSE:", core.UNIVERSE)
print("TF:", core.TF_API, "| Donchian masuk/keluar:", core.ENTRY_N, core.EXIT_K, "| SMA:", core.SMA_M)
print("Maks posisi:", core.MAX_POSITIONS, "| per posisi:", core.MAX_POS_FRAC, "| loss limit:", core.DAILY_LOSS_LIMIT)

In [ ]:
# SEL 2: cek bentuk respons history_v2 (1 request). Kirim hasil cetaknya ke Claude bila ada yang aneh.
import time, json, urllib.request
BASE = "https://indodax.com"
def get(url, tries=3):
    last = None
    for i in range(tries):
        try:
            time.sleep(0.4)   # sekitar 2,5 request/detik, tidak paralel
            req = urllib.request.Request(url, headers={"User-Agent": "uas-backtest/1.0"})
            with urllib.request.urlopen(req, timeout=20) as r:
                return json.loads(r.read().decode())
        except Exception as e:
            last = e; time.sleep(2 * (i + 1))
    raise RuntimeError("gagal: %s" % last)

now = core.to_seconds(get(BASE + "/api/server_time")["server_time"])
raw = get("%s/tradingview/history_v2?symbol=BTCIDR&tf=%s&from=%d&to=%d" % (BASE, core.TF_API, now - 7*86400, now))
print("tipe:", type(raw).__name__, "| panjang:", len(raw))
items = raw if isinstance(raw, list) else raw.get("data", raw)
print("contoh 2 pertama:", items[:2] if isinstance(items, list) else items)
print("contoh terakhir:", items[-1] if isinstance(items, list) else "")
print("hasil normalisasi:", core.normalize_bars(raw)[:2])


In [ ]:
# SEL 3: tarik candle 4 jam ~3 tahun untuk semua pair, simpan ke CSV
import csv
from datetime import datetime, timezone
DATA_START = "2023-09-01"      # pemanasan indikator; evaluasi backtest mulai 2023-11-01
CHUNK_DAYS = 90
start_ts = int(datetime.fromisoformat(DATA_START).replace(tzinfo=timezone.utc).timestamp())
now = core.to_seconds(get(BASE + "/api/server_time")["server_time"])
for pair in core.UNIVERSE:
    bars, t = [], start_ts
    while t < now:
        to = min(t + CHUNK_DAYS * 86400, now)
        raw = get("%s/tradingview/history_v2?symbol=%s&tf=%s&from=%d&to=%d" % (BASE, pair.upper(), core.TF_API, t, to))
        bars += core.normalize_bars(raw)
        t = to
    uniq = {b["t"]: b for b in bars}
    bars = core.drop_open_bars(sorted(uniq.values(), key=lambda b: b["t"]), now)
    with open("data_%s.csv" % pair, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["t", "o", "h", "l", "c", "v"]); w.writeheader(); w.writerows(bars)
    if not bars:
        print(pair, ": tidak ada data (cek nama pair / respons API)"); continue
    expected = (bars[-1]["t"] - bars[0]["t"]) // core.TF_SECONDS + 1
    print(pair, "| bar:", len(bars), "| harapan:", expected, "| hilang: %.1f%%" % (100 * (1 - len(bars) / expected)))
    print("   dari", datetime.fromtimestamp(bars[0]["t"], timezone.utc), "sampai", datetime.fromtimestamp(bars[-1]["t"], timezone.utc))


In [ ]:
# SEL 4: ambil log bot dari Redis, simpan mentah, hitung median spread & slippage per pair
# Pakai token READ-ONLY dari Vercel (KV_REST_API_READ_ONLY_TOKEN) dan URL (KV_REST_API_URL). Input disembunyikan.
from getpass import getpass
import statistics
UP_URL = getpass("URL Redis: ").strip()
UP_TOK = getpass("Token read-only: ").strip()
def rcmd(cmd):
    req = urllib.request.Request(UP_URL, data=json.dumps(cmd).encode(),
                                 headers={"Authorization": "Bearer " + UP_TOK, "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]
n = rcmd(["LLEN", "bot:log"])
logs = []
for s in range(0, n, 500):
    logs += [json.loads(x) for x in rcmd(["LRANGE", "bot:log", s, min(s + 499, n - 1)])]
open("bot_log.jsonl", "w").write("\n".join(json.dumps(x) for x in logs))
print("entri log:", len(logs))
snaps = {}
for e in logs:
    if e.get("type") == "snapshot":
        snaps.setdefault(e["pair"], []).append(e)
params = {}
for pair in core.UNIVERSE:
    s = snaps.get(pair, [])
    if not s:
        print(pair, ": belum ada snapshot"); continue
    sp = statistics.median(x["spread_pct"] for x in s)
    sl = [x["buy_slip_pct_ref20jt"] for x in s if x.get("buy_slip_pct_ref20jt") is not None]
    params[pair] = {"spread_pct": sp, "slip_pct": statistics.median(sl) if sl else 0.0, "n": len(s)}
    print(pair, "| snapshot:", len(s), "| median spread %.4f%% | median slippage %.4f%%" % (sp, params[pair]["slip_pct"]))
json.dump(params, open("bt_params.json", "w"), indent=1)
if any(v["n"] < 30 for v in params.values()):
    print("CATATAN: snapshot masih sedikit (<30). Ulangi sel ini nanti supaya median lebih stabil.")


In [ ]:
# SEL 5: backtest (memakai fungsi sinyal, validasi, dan pembulatan yang SAMA dengan bot di _core.py)
import csv, json, math, statistics
from datetime import datetime, timezone

FEE = 0.0041555          # sama dengan FEE_TAKER di Vercel (taker all-in, dipakai untuk beli dan jual)
BT_START = "2023-11-01"  # awal evaluasi (data mulai 2023-09-01 untuk pemanasan)
BT_END = None            # None = sampai data terakhir

def to_ts(s):
    d = datetime.fromisoformat(s)
    if d.tzinfo is None:
        d = d.replace(tzinfo=timezone.utc)
    return int(d.timestamp())

def load(pair):
    rows = []
    with open("data_%s.csv" % pair) as f:
        for r in csv.DictReader(f):
            rows.append({k: (int(float(v)) if k == "t" else float(v)) for k, v in r.items()})
    return rows

def run_backtest(start_iso, end_iso=None, fee=FEE, verbose=True):
    try:
        params = json.load(open("bt_params.json"))
    except FileNotFoundError:
        raise SystemExit("bt_params.json belum ada: jalankan SEL 4 dulu (spread & slippage harus dari snapshot, bukan tebakan).")
    for p in core.UNIVERSE:
        if p not in params:
            raise SystemExit("Belum ada snapshot untuk %s; jalankan SEL 4 lagi setelah data terkumpul." % p)
    TF = core.TF_SECONDS
    t0 = to_ts(start_iso); t1 = to_ts(end_iso) if end_iso else 10**12
    data = {p: load(p) for p in core.UNIVERSE}
    idx = {p: {b["t"]: i for i, b in enumerate(data[p])} for p in data}
    half = {p: params[p]["spread_pct"] / 200.0 for p in data}
    slip = {p: params[p]["slip_pct"] / 100.0 for p in data}
    events = sorted({b["t"] for p in data for b in data[p] if t0 <= b["t"] + TF <= t1})
    cash = float(core.INITIAL_CASH); pos = {}; trades = []; eq = []
    last_close = {}; day = None; day_start = cash; loss_logged = None
    stat = {"bar_dilewati_data_tidak_valid": 0, "buy_diblok_loss_limit": 0, "buy_diblok_maks_posisi": 0, "buy_diblok_min_order": 0}

    def mtm():
        return cash + sum(q["qty"] * last_close.get(p, q["entry_ref"]) * (1 - half[p]) for p, q in pos.items())

    for t in events:
        ct = t + TF
        for p in data:
            i = idx[p].get(t)
            if i is not None:
                last_close[p] = data[p][i]["c"]
        equity_pre = mtm()
        d = core.wib_date(ct)
        if d != day:
            day, day_start = d, equity_pre
        loss_hit = equity_pre < day_start * (1 - core.DAILY_LOSS_LIMIT)
        for p in core.UNIVERSE:
            i = idx[p].get(t)
            if i is None or i + 1 >= len(data[p]) or data[p][i + 1]["t"] != t + TF:
                continue
            win = data[p][max(0, i - core.BARS_NEEDED + 1): i + 1]
            ok, why = core.validate(win, [(1.0, 1.0)], [(1.0, 1.0)], ct)   # cek bar hilang/nol, sama dgn bot
            if not ok:
                stat["bar_dilewati_data_tidak_valid"] += 1
                continue
            sig = core.compute_signal(win, p in pos)
            ref = data[p][i + 1]["o"]          # harga acuan eksekusi = open bar berikutnya
            if sig["signal"] == "SELL":
                q = pos.pop(p)
                price = ref * (1 - half[p] - slip[p])
                gross = q["qty"] * price; fee_out = gross * fee
                cash += gross - fee_out
                trades.append({"pair": p, "entry_t": q["entry_t"], "exit_t": ct, "qty": q["qty"],
                               "entry_price": q["entry_price"], "exit_price": price,
                               "pnl": gross - fee_out - q["cost_total"],
                               "fee": q["fee_in"] + fee_out,
                               "spread_slip": q["slip_in"] + q["qty"] * (ref - price),
                               "gross_before_costs": (ref - q["entry_ref"]) * q["qty"]})
            elif sig["signal"] == "BUY":
                size = min(core.MAX_POS_FRAC * equity_pre, cash)
                if loss_hit:
                    stat["buy_diblok_loss_limit"] += 1; continue
                if len(pos) >= core.MAX_POSITIONS:
                    stat["buy_diblok_maks_posisi"] += 1; continue
                if size < core.MIN_ORDER_IDR:
                    stat["buy_diblok_min_order"] += 1; continue
                price = ref * (1 + half[p] + slip[p])
                qty = core.floor_qty(size / (1 + fee) / price)
                cost = qty * price
                if qty <= 0 or cost < core.MIN_ORDER_IDR:
                    stat["buy_diblok_min_order"] += 1; continue
                fee_in = cost * fee
                cash -= cost + fee_in
                pos[p] = {"qty": qty, "entry_t": ct, "entry_price": price, "entry_ref": ref,
                          "cost_total": cost + fee_in, "fee_in": fee_in, "slip_in": qty * (price - ref)}
        eq.append((ct, mtm()))
    return eq, trades, stat, pos, data, t0, t1

def metrics(eq, trades, stat, pos, data, t0, t1, long_window=True):
    e0 = float(core.INITIAL_CASH)
    vals = [v for _, v in eq]
    peak, mdd = vals[0], 0.0
    for v in vals:
        peak = max(peak, v); mdd = min(mdd, v / peak - 1)
    years = (eq[-1][0] - eq[0][0]) / (365.25 * 86400)
    res = {"return_total_pct": (vals[-1] / e0 - 1) * 100, "max_drawdown_pct": mdd * 100,
           "equity_akhir": vals[-1], "jumlah_trade": len(trades), "posisi_masih_terbuka": list(pos)}
    if long_window and years > 0.5:
        res["cagr_pct"] = ((vals[-1] / e0) ** (1 / years) - 1) * 100
    pnl = [t["pnl"] for t in trades]
    wins = [x for x in pnl if x > 0]; losses = [x for x in pnl if x <= 0]
    res["win_rate_pct"] = 100 * len(wins) / len(pnl) if pnl else None
    res["rata2_untung"] = statistics.mean(wins) if wins else None
    res["rata2_rugi"] = statistics.mean(losses) if losses else None
    res["profit_factor"] = (sum(wins) / abs(sum(losses))) if losses and sum(losses) != 0 else None
    gross = sum(t["gross_before_costs"] for t in trades)
    costs = sum(t["fee"] + t["spread_slip"] for t in trades)
    res["porsi_biaya"] = (costs / gross) if gross > 0 else None
    res["total_fee"] = sum(t["fee"] for t in trades); res["total_spread_slippage"] = sum(t["spread_slip"] for t in trades)
    # benchmark buy and hold BTC/IDR pada periode yang sama (tanpa fee)
    btc = data["btcidr"]
    bars = [b for b in btc if t0 <= b["t"] + core.TF_SECONDS <= t1]
    if bars:
        res["buy_hold_btc_pct"] = (bars[-1]["c"] / bars[0]["o"] - 1) * 100
    res.update(stat)
    return res

eq, trades, stat, pos, data, t0, t1 = run_backtest(BT_START, BT_END)
res = metrics(eq, trades, stat, pos, data, t0, t1)
for k, v in res.items():
    print("%-30s %s" % (k, ("%.4f" % v) if isinstance(v, float) else v))
with open("bt_trades.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(trades[0].keys()) if trades else ["pair"]); w.writeheader(); w.writerows(trades)
with open("bt_equity.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["t_unix", "equity"]); w.writerows(eq)
json.dump(res, open("bt_result.json", "w"), indent=1, default=str)
print("\nTersimpan: bt_result.json, bt_trades.csv, bt_equity.csv")


In [ ]:
# SEL 6: kemas hasil untuk diunduh
import glob, zipfile
from google.colab import files
with zipfile.ZipFile("hasil_backtest.zip", "w") as z:
    for f in glob.glob("data_*.csv") + glob.glob("bt_*") + ["bot_log.jsonl", "_core.py"]:
        z.write(f)
files.download("hasil_backtest.zip")
